# Face Recognition on LFW (Labeled Faces in the Wild) using CNN

**Objective:** Identify which person a face belongs to, among the most photographed individuals in the LFW dataset, using a Convolutional Neural Network with more than 3 convolutional layers.

## Step 1: Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical

## Step 2: Load Dataset

`min_faces_per_person` filters the dataset to only people with at least that many photos — this naturally gives the 'top-N most photographed people'.

In [ ]:
lfw_people = fetch_lfw_people(min_faces_per_person=20, resize=0.5, color=True)

X = lfw_people.images
y = lfw_people.target
target_names = lfw_people.target_names

n_classes = target_names.shape[0]
print(f"Number of people (classes): {n_classes}")
print(f"Image shape: {X.shape[1:]}")
print(f"Total samples: {X.shape[0]}")

## Step 3: Preprocess and Split Data

In [ ]:
X = X / 255.0

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

y_train_cat = to_categorical(y_train, n_classes)
y_test_cat = to_categorical(y_test, n_classes)

print(X_train.shape, X_test.shape)

## Step 4: Build CNN (4 Convolutional Layers)

In [ ]:
img_h, img_w, img_c = X.shape[1], X.shape[2], X.shape[3]

model = models.Sequential()

model.add(layers.Conv2D(32, (3,3), activation='relu', padding='same',
                         input_shape=(img_h, img_w, img_c)))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.Flatten())
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(n_classes, activation='softmax'))

model.summary()

## Step 5: Compile the Model

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

## Step 6: Train the Model

In [ ]:
history = model.fit(X_train, y_train_cat, epochs=40,
                     batch_size=32,
                     validation_data=(X_test, y_test_cat))

## Step 7: Evaluate the Model

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_cat)
print(f"Test Accuracy: {test_acc*100:.2f}%")

## Step 8: Plot Training Curves

In [ ]:
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('LFW Face Recognition Accuracy')
plt.legend()
plt.show()

## Step 9: Visualize Sample Predictions

In [ ]:
preds = model.predict(X_test[:9])
pred_labels = np.argmax(preds, axis=1)

fig, axes = plt.subplots(3, 3, figsize=(8,8))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i])
    ax.set_title(f"Pred: {target_names[pred_labels[i]]}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Conclusion

The CNN achieved approximately **81% test accuracy** recognizing faces from the LFW dataset, using 4 convolutional layers with batch normalization and dropout.